# 📊 Pipeline de Preparação de Dados — Passos Mágicos (2022-2024)

## 🎯 Objetivo

Consolidar e padronizar os dados educacionais da Associação Passos Mágicos (2022-2024) para criar um dataset estruturado pronto para modelagem preditiva de risco de defasagem escolar.

---

## 🧠 Lógica de Negócio

### 1. Desafios Identificados

| Desafio | Descrição | Solução |
|---------|-----------|---------|
| **Heterogeneidade de Colunas** | Cada ano tem colunas e nomenclaturas diferentes | Padronização com sufixo de ano (_22, _23, _24) |
| **Alunos com Múltiplos Registros** | Mesma pessoa em diferentes fases/turmas | Agregação por ID único (NOME como chave) |
| **Valores Ausentes** | Alto índice de nulos em features comportamentais | Análise de cobertura + imputação controlada |
| **Fase Inválida ("9")** | "9" válido em 2024, mas não em 2022/2023 | Validação com ano de contexto |
| **Inconsistência de Tipo** | Colunas numéricas como string | Conversão explícita com tratamento de erro |

### 2. Estratégia de Consolidação

```mermaid
graph LR
    A["📂 Dados Brutos<br/>2022, 2023, 2024"] --> B["🔄 Padronizar<br/>Nomenclatura + Sufixo Ano"]
    B --> C["➕ Adicionar<br/>Colunas Vazias"]
    C --> D["🏷️ Transform<br/>FASE, TURMA, Flags"]
    D --> E["🔗 Consolidar<br/>em Dataset Único"]
    E --> F["✅ QA: Nulos,<br/>Tipos, Ranges"]
    F --> G["💾 Salvar Parquet<br/>dataset_consolidado.parquet"]
```

### 3. Transformações Principais por Módulo

#### 📄 **scripts/data_processing.py** (ETL Core)
- `padronizar_colunas_ano(df, ano)` — Renomeia colunas com sufixo do ano
- `adicionar_colunas_vazias(df1, df2)` — Sincroniza estrutura entre years
- `consolidar_dataframes([df1, df2, df3])` — Union com alinhamento de tipos
- `analise_nulos(df)` — Relatório de cobertura por coluna

#### 🏷️ **scripts/notebook_feature_engineering.py** (Domain Transformations)
- `transformar_fase(col, ano)` — Converte FASE para formato Passos Mágicos (ALFA, 1A-9, etc)
- `derivar_turma(fase, ano_ingresso)` — Calcula TURMA esperada vs real
- `criar_flags_derivadas(df)` — Gera VETERANO, EM_FASE, etc

#### 📊 **scripts/visualization.py** (Analysis)
- `plot_timeline_cobertura(df_list)` — Mostra % de preenchimento por year
- `analisa_continuidade(df_2022, df_2023)` — Alunos que aparecem em múltiplos years

### 4. Invariantes Garantidas ✅

| Garantia | Implementação |
|----------|----------------|
| **Imutabilidade** | Todas as funções retornam `.copy()` |
| **Rastreabilidade** | Sufixo `_XX` identifica source year |
| **Reprodutibilidade** | Sem random states / ordem determinística |
| **Type Safety** | Explicit typing em annotations |
| **Testabilidade** | 5+ unit tests por função em `tests/test_data_processing.py` |

### 5. Métricas de Qualidade Esperadas

| KPI | Target | Threshold |
|-----|--------|-----------|
| **Nulos por coluna** | < 30% | Aviso se > 50% |
| **Alunos únicos** | 700-900 | Flag se < 600 ou > 1000 |
| **Colunas após consolidação** | 80-120 | Depende de fields comuns |
| **Registros duplicados** | 0 | Falha se > 0 |
| **Inconsistência de tipos** | 0 | Converção ou warning |

---

## 🔗 Arquitetura (apenas importações neste notebook)

Este notebook **orquestra** o pipeline importando funções testadas de:

- **`scripts/data_processing.py`** — Funções de ETL puro
- **`scripts/notebook_feature_engineering.py`** — Transformações de domínio
- **`pandas`, `numpy`** — Manipulação de dados
- **Output**: `app/data/processed/dataset_consolidado_2022_2024.parquet`

Para detalhes de implementação, veja docstrings nas funções importadas.

In [ ]:
# Imports necessários
import sys
from pathlib import Path

import pandas as pd

# Adiciona diretórios ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports de módulos do projeto
from scripts.data_processing import (
    adicionar_colunas_vazias,
    analise_nulos,
    consolidar_dataframes,
    padronizar_colunas_ano,
)
from scripts.notebook_feature_engineering import (
    aplicar_transformacoes_fase_turma,
    criar_coluna_em_fase,
    criar_coluna_veterano,
)

# Configurações
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("✅ Imports realizados com sucesso")
print(f"📁 Project root: {project_root}")

---

## 📂 Etapa 1: Carregamento de Dados Brutos

Carrega arquivos Excel de cada ano.

In [ ]:
# Define caminhos dos arquivos
data_dir = project_root / "app" / "data" / "raw"
arquivo_2022 = data_dir / "BASE DE DADOS PEDE 2022 - DATATHON.xlsx"
arquivo_2023 = data_dir / "BASE DE DADOS PEDE 2023 - DATATHON.xlsx"
arquivo_2024 = data_dir / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# Carrega dados
df_2022 = pd.read_excel(arquivo_2022)
df_2023 = pd.read_excel(arquivo_2023)
df_2024 = pd.read_excel(arquivo_2024)

print("📊 Dimensões originais:")
print(f"  2022: {df_2022.shape[0]:>4} linhas × {df_2022.shape[1]:>3} colunas")
print(f"  2023: {df_2023.shape[0]:>4} linhas × {df_2023.shape[1]:>3} colunas")
print(f"  2024: {df_2024.shape[0]:>4} linhas × {df_2024.shape[1]:>3} colunas")

---

## 🔧 Etapa 2: Padronização de Nomenclatura

Aplica padrão `{NOME_COLUNA}_{ANO}` em maiúsculas para todas as colunas.

In [ ]:
# Padroniza colunas por ano
df_2022_pad = padronizar_colunas_ano(df_2022, 2022, ignorar_cols=["NOME"])
df_2023_pad = padronizar_colunas_ano(df_2023, 2023, ignorar_cols=["NOME"])
df_2024_pad = padronizar_colunas_ano(df_2024, 2024, ignorar_cols=["NOME"])

print("✅ Colunas padronizadas")
print("\n📋 Exemplo de colunas 2022 (primeiras 10):")
print(list(df_2022_pad.columns[:10]))

---

## 🧩 Etapa 3: Uniformização de Estrutura

Adiciona colunas ausentes com valores NaN para garantir mesma estrutura entre anos.

In [ ]:
# Define conjunto de colunas esperadas (união de todas)
todas_colunas = set(df_2022_pad.columns) | set(df_2023_pad.columns) | set(df_2024_pad.columns)

# Adiciona colunas faltantes em cada DataFrame
df_2022_uni = adicionar_colunas_vazias(df_2022_pad, list(todas_colunas))
df_2023_uni = adicionar_colunas_vazias(df_2023_pad, list(todas_colunas))
df_2024_uni = adicionar_colunas_vazias(df_2024_pad, list(todas_colunas))

print(f"✅ Estrutura uniformizada: {len(todas_colunas)} colunas totais")
print(
    f"  2022: {df_2022_uni.shape[1]} colunas (adicionadas {df_2022_uni.shape[1] - df_2022_pad.shape[1]})"
)
print(
    f"  2023: {df_2023_uni.shape[1]} colunas (adicionadas {df_2023_uni.shape[1] - df_2023_pad.shape[1]})"
)
print(
    f"  2024: {df_2024_uni.shape[1]} colunas (adicionadas {df_2024_uni.shape[1] - df_2024_pad.shape[1]})"
)

---

## 🎭 Etapa 4: Transformação de FASE e TURMA

Normaliza códigos de fase (ALFA, 8A, etc) para formato padrão.

**Atenção:** Regras de 2024 diferem de 2022/2023 (fase '9').

In [ ]:
# Identifica coluna de fase em cada ano
col_fase_2022 = "FASE_22" if "FASE_22" in df_2022_uni.columns else None
col_fase_2023 = "FASE_23" if "FASE_23" in df_2023_uni.columns else None
col_fase_2024 = "FASE_24" if "FASE_24" in df_2024_uni.columns else None

# Aplica transformações específicas por ano
if col_fase_2022:
    df_2022_trans = aplicar_transformacoes_fase_turma(df_2022_uni, col_fase_2022, ano=2022)
else:
    df_2022_trans = df_2022_uni.copy()

if col_fase_2023:
    df_2023_trans = aplicar_transformacoes_fase_turma(df_2023_uni, col_fase_2023, ano=2023)
else:
    df_2023_trans = df_2023_uni.copy()

if col_fase_2024:
    df_2024_trans = aplicar_transformacoes_fase_turma(df_2024_uni, col_fase_2024, ano=2024)
else:
    df_2024_trans = df_2024_uni.copy()

print("✅ Transformações de FASE e TURMA aplicadas")
if col_fase_2022:
    print("\nDistribuição de FASE_PADRONIZADA (2022):")
    print(df_2022_trans["FASE_PADRONIZADA"].value_counts())

---

## 🏗️ Etapa 5: Criação de Features Derivadas

Cria flags e variáveis calculadas a partir dos dados brutos.

In [ ]:
# Cria coluna VETERANO (ingressou antes de 2022)
if "ANO_INGRESSO_22" in df_2022_trans.columns:
    df_2022_feat = criar_coluna_veterano(df_2022_trans, "ANO_INGRESSO_22", ano_corte=2022)
else:
    df_2022_feat = df_2022_trans.copy()

# Cria coluna EM_FASE (está na fase ideal)
if "FASE_PADRONIZADA" in df_2022_feat.columns and "FASE_IDEAL_22" in df_2022_feat.columns:
    df_2022_feat = criar_coluna_em_fase(df_2022_feat, "FASE_PADRONIZADA", "FASE_IDEAL_22")

print("✅ Features derivadas criadas")
if "VETERANO" in df_2022_feat.columns:
    print("\nDistribuição VETERANO:")
    print(df_2022_feat["VETERANO"].value_counts(normalize=True) * 100)

---

## 🔗 Etapa 6: Consolidação dos DataFrames

Merge dos 3 anos usando NOME como chave.

In [ ]:
# Consolida DataFrames
df_consolidado = consolidar_dataframes(
    [df_2022_feat, df_2023_trans, df_2024_trans], id_col="NOME", sufixos=["_2022", "_2023", "_2024"]
)

print("✅ Consolidação concluída")
print(f"  Dimensão final: {df_consolidado.shape[0]} linhas × {df_consolidado.shape[1]} colunas")
print(f"  Alunos únicos: {df_consolidado['NOME'].nunique()}")

---

## 📊 Etapa 7: Análise de Qualidade

Diagnóstico de valores ausentes e tipos de dados.

In [ ]:
# Análise de nulos
relatorio_nulos = analise_nulos(df_consolidado)

print("📊 Top 10 colunas com mais valores ausentes:\n")
print(relatorio_nulos.head(10))

# Estatísticas gerais
print("\n📈 Estatísticas de qualidade:")
print(f"  Total de colunas: {len(df_consolidado.columns)}")
print(f"  Colunas com nulos: {len(relatorio_nulos)}")
print(f"  % médio de nulos (colunas afetadas): {relatorio_nulos['perc_nulos'].mean():.2f}%")

---

## 💾 Etapa 8: Exportação

Salva dataset consolidado para uso posterior.

In [ ]:
# Define caminho de saída
output_dir = project_root / "app" / "data" / "processed"
output_dir.mkdir(exist_ok=True, parents=True)

output_file = output_dir / "dataset_consolidado_2022_2024.parquet"

# Salva em formato Parquet (mais eficiente que CSV)
df_consolidado.to_parquet(output_file, index=False)

print("✅ Dataset consolidado salvo em:")
print(f"  {output_file}")
print(f"\n📦 Tamanho do arquivo: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

---

## ✅ Conclusão

### Entregáveis
- ✅ Dataset consolidado de 2022-2024
- ✅ Features padronizadas e normalizadas
- ✅ Variáveis derivadas criadas (VETERANO, EM_FASE)
- ✅ Relatório de qualidade de dados

### Próximos Passos
1. **EDA detalhada** → `eda_passos_magicos.ipynb`
2. **Feature Engineering** → Engenharia avançada de features
3. **Modelagem** → Treinamento de modelos preditivos

### Referências
- Funções: `scripts/data_processing.py`, `scripts/notebook_feature_engineering.py`
- Testes: `tests/test_data_processing.py`, `tests/test_notebook_feature_engineering.py`
- Documentação: [README.md](../README.md)
